# Aspectos preliminares

Este script tiene por objetivo verificar el contenido de los datasets luego de la normalización.

In [11]:
# Importacion de librerias.
import pandas as pd
import numpy as np
import os
from pandasql import sqldf

In [12]:
# Inicializacion de variables.
lookup_tables = ['application_train', 'bureau', 'previous_application']
lookup_dir = '../output/'

# Ejecución del control

In [13]:
# Función para cargar DataFrames desde archivos CSV en una carpeta específica
def load_dataframes(folder_path, table_name):
    """
    Carga DataFrames desde archivos CSV en la carpeta especificada.
    
    Args:
        folder_path (str): Ruta a la carpeta base (ej. '../output/').
        table_name (str): Nombre de la subcarpeta (ej. 'application_train').
    
    Returns:
        dict: Diccionario con nombres de archivos (sin extensión) como claves y DataFrames como valores.
    """
    full_path = os.path.join(folder_path, table_name)
    
    # Verificar si la carpeta existe
    if not os.path.exists(full_path):
        print(f"La carpeta {full_path} no existe.")
        return {}
    
    # Listar archivos CSV en la carpeta
    csv_files = [f for f in os.listdir(full_path) if f.endswith('.csv')]
    
    if not csv_files:
        print("No se encontraron archivos CSV en la carpeta.")
        return {}
    
    # Diccionario para almacenar los DataFrames
    dataframes = {}
    
    # Leer cada archivo CSV y crear un DataFrame
    for file in csv_files:
        file_path = os.path.join(full_path, file)
        try:
            df = pd.read_csv(file_path)
            # Usar nombre del archivo sin extensión como clave
            key = file.replace('.csv', '')
            dataframes[key] = df
            print(f"DataFrame creado para '{key}' con shape {df.shape}.")
        except Exception as e:
            print(f"Error al leer '{file}': {e}")
    
    # Opcional: Mostrar un resumen de los DataFrames creados
    print("\nResumen de DataFrames creados:")
    for name, df in dataframes.items():
        print(f"- {name}: {df.shape[0]} filas, {df.shape[1]} columnas")
    
    return dataframes

In [14]:
# Llamada a la función para cargar los DataFrames
application_train_dataframes = load_dataframes(lookup_dir, lookup_tables[0])

DataFrame creado para 'application_train' con shape (307511, 123).
DataFrame creado para 'dim_code_gender' con shape (3, 2).
DataFrame creado para 'dim_emergencystate_mode' con shape (2, 2).
DataFrame creado para 'dim_flag_own_car' con shape (2, 2).
DataFrame creado para 'dim_flag_own_realty' con shape (2, 2).
DataFrame creado para 'dim_fondkapremont_mode' con shape (4, 2).
DataFrame creado para 'dim_housetype_mode' con shape (3, 2).
DataFrame creado para 'dim_name_contract_type' con shape (2, 2).
DataFrame creado para 'dim_name_education_type' con shape (5, 2).
DataFrame creado para 'dim_name_family_status' con shape (6, 2).
DataFrame creado para 'dim_name_housing_type' con shape (6, 2).
DataFrame creado para 'dim_name_income_type' con shape (8, 2).
DataFrame creado para 'dim_name_type_suite' con shape (7, 2).
DataFrame creado para 'dim_occupation_type' con shape (18, 2).
DataFrame creado para 'dim_organization_type' con shape (35, 2).
DataFrame creado para 'dim_organization_type_2' c

In [15]:
# Head de muestra.
application_train_dataframes['application_train'].head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE_ID,CODE_GENDER_ID,FLAG_OWN_CAR_ID,FLAG_OWN_REALTY_ID,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,ORGANIZATION_TYPE_2_ID
0,100002,1,1,1,1,1,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1
1,100003,0,1,2,1,2,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2
2,100004,0,2,1,2,1,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2
3,100006,0,1,2,1,1,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,1
4,100007,0,1,1,1,1,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,2


In [ ]:


# Query para la verificación del join de tablas principal y de dimensiones.
query = f"""
SELECT {', '.join(df_backup.columns)}
FROM df
"""

dim_tables_names = [*dimension_tables.keys()]

for dim_table_name in dim_tables_names:
  dim_table_id = dimension_tables[dim_table_name].columns[0]

  query += f"""
  LEFT JOIN {dim_table_name} ON df.{dim_table_id} = {dim_table_name}.{dim_table_id}
  """

In [32]:
# Query diseñada para unir application_train con las tablas de dimensión, reemplazando IDs por descripciones

# Identificar la tabla principal y las tablas de dimensión
main_table = 'application_train'
dim_tables = {k: v for k, v in application_train_dataframes.items() if k.startswith('dim_')}

# Obtener columnas de la tabla principal
main_columns = list(application_train_dataframes[main_table].columns)

# Construir la lista de columnas para SELECT, reemplazando IDs por descripciones
select_columns = []
for col in main_columns:
    if col.endswith('_ID'):
        # Encontrar la tabla de dimensión correspondiente
        dim_base = col.replace('_ID', '').lower()
        dim_name = f"dim_{dim_base}"
        if dim_name in dim_tables:
            # Usar la segunda columna (descripción) de la dim table
            desc_col = application_train_dataframes[dim_name].columns[1]
            select_columns.append(f"{dim_name}.{desc_col} AS {desc_col}")
        else:
            select_columns.append(f"{main_table}.{col}")
    else:
        select_columns.append(f"{main_table}.{col}")

# Construir la query SQL
query = f"SELECT {', '.join(select_columns)} FROM {main_table}"

for dim_name in dim_tables:
    # Asumir que el ID es la primera columna de cada tabla de dimensión
    id_col = application_train_dataframes[dim_name].columns[0]
    query += f" LEFT JOIN {dim_name} ON {main_table}.{id_col} = {dim_name}.{id_col}"

# Ejecutar la query usando pandasql
result = sqldf(query, application_train_dataframes)

# Mostrar el resultado (primeras filas para comparación)
result.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,ORGANIZATION_TYPE_2
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,Type 3
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,Type 3
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,Type 0


In [ ]:
# Lectura del df original para comparación.
original_df = pd.read_csv('../data/application_train.csv')

In [35]:
# Control de shape.
print("Shape original:", original_df.shape)
print("Shape resultante:", result.shape)

Shape original: (307511, 122)
Shape resultante: (307511, 123)


In [37]:
# Verificación de columnas en original_df.
for col in original_df.columns:
    if col in result.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

# Verificación de columnas en result.
for col in result.columns:
    if col in original_df.columns:
        continue
    else:
        print(f"Columna '{col}' NO está presente en el resultado.")

Columna 'ORGANIZATION_TYPE_2' NO está presente en el resultado.


In [45]:
# Script para comparar el contenido columna por columna entre original_df y result, excluyendo ORGANIZATION_TYPE y ORGANIZATION_TYPE_2

# Columnas a excluir
exclude_columns = ['ORGANIZATION_TYPE', 'ORGANIZATION_TYPE_2']

# Obtener columnas comunes, excluyendo las especificadas
common_columns = [col for col in original_df.columns if col not in exclude_columns]

print("Comparación columna por columna entre original_df y result:\n")

all_match = True

for col in common_columns:
    if col in result.columns:
        # Verificar si es una columna de ID (termina con '_ID')
        if col.endswith('_ID'):
            # Asumir que está correctamente reemplazada, no marcar como no coincidente
            continue
        else:
            # Comparar valores directamente
            if not original_df[col].equals(result[col]):
                print(f"Columna '{col}' no coincide.")
                all_match = False
    else:
        print(f"Columna '{col}' no está presente en result.")
        all_match = False

if all_match:
    print("Todas las columnas coinciden.")

print("\nComparación completada.")

Comparación columna por columna entre original_df y result:

Todas las columnas coinciden.

Comparación completada.


In [70]:
# Detalle de diferencias encontradas.
for col in original_df.columns:
    if col == 'ORGANIZATION_TYPE' or col == 'ORGANIZATION_TYPE_2':
        continue
    else:   
        differences = sum(original_df[col] != result[col])
        if differences > 0:
            print(f"Columna '{col}' tiene {differences} diferencias.", end=' ')
            print(original_df[original_df[col] != result[col]][col].value_counts(dropna=False).index[0])
        else:
            continue


Columna 'AMT_ANNUITY' tiene 12 diferencias. nan
Columna 'AMT_GOODS_PRICE' tiene 278 diferencias. nan
Columna 'NAME_TYPE_SUITE' tiene 1292 diferencias. nan
Columna 'OWN_CAR_AGE' tiene 202929 diferencias. nan
Columna 'OCCUPATION_TYPE' tiene 96391 diferencias. nan
Columna 'CNT_FAM_MEMBERS' tiene 2 diferencias. nan
Columna 'EXT_SOURCE_1' tiene 173378 diferencias. nan
Columna 'EXT_SOURCE_2' tiene 660 diferencias. nan
Columna 'EXT_SOURCE_3' tiene 60965 diferencias. nan
Columna 'APARTMENTS_AVG' tiene 156061 diferencias. nan
Columna 'BASEMENTAREA_AVG' tiene 179943 diferencias. nan
Columna 'YEARS_BEGINEXPLUATATION_AVG' tiene 150007 diferencias. nan
Columna 'YEARS_BUILD_AVG' tiene 204488 diferencias. nan
Columna 'COMMONAREA_AVG' tiene 214865 diferencias. nan
Columna 'ELEVATORS_AVG' tiene 163891 diferencias. nan
Columna 'ENTRANCES_AVG' tiene 154828 diferencias. nan
Columna 'FLOORSMAX_AVG' tiene 153020 diferencias. nan
Columna 'FLOORSMIN_AVG' tiene 208642 diferencias. nan
Columna 'LANDAREA_AVG' ti